In [63]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [64]:
df = pd.read_csv('car_fuel_efficiency.csv')

In [65]:
df

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors,fuel_efficiency_mpg
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0,13.231729
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0,13.688217
2,170,NaN,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0,14.246341
3,220,4.0,NaN,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0,16.912736
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0,12.488369
...,...,...,...,...,...,...,...,...,...,...,...
9699,140,5.0,164.0,2981.107371,17.3,2013,Europe,Diesel,Front-wheel drive,NaN,15.101802
9700,180,NaN,154.0,2439.525729,15.0,2004,USA,Gasoline,All-wheel drive,0.0,17.962326
9701,220,2.0,138.0,2583.471318,15.1,2008,USA,Diesel,All-wheel drive,-1.0,17.186587
9702,230,4.0,177.0,2905.527390,19.4,2011,USA,Diesel,Front-wheel drive,1.0,15.331551


In [66]:
df = df.fillna(0)

In [67]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['fuel_efficiency_mpg']
y_val = df_val['fuel_efficiency_mpg']
y_test = df_test['fuel_efficiency_mpg']

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [68]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')
test_dict = df_test.to_dict(orient='records')

dv = DictVectorizer(sparse=False)

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)
X_test = dv.transform(test_dict)

dt = DecisionTreeRegressor(random_state=1)
dt.fit(X_train, y_train)

y_pred_val = dt.predict(X_val)
rmse_val = root_mean_squared_error(y_val, y_pred_val)
print("Validation RMSE:", rmse_val)

y_pred_test = dt.predict(X_test)
rmse_test = root_mean_squared_error(y_test, y_pred_test)
print("Test RMSE:", rmse_test)


Validation RMSE: 0.6172416932409205
Test RMSE: 0.6026528842178787


In [69]:
feature_names = dv.get_feature_names_out()

root_feature_index = dt.tree_.feature[0]
root_feature_name = feature_names[root_feature_index]

print("Feature used for splitting:", root_feature_name)

Feature used for splitting: vehicle_weight


In [70]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)

rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)

rmse_val = root_mean_squared_error(y_val, y_pred_val)
print("Validation RMSE:", rmse_val)

Validation RMSE: 0.4599777557336148


In [76]:
for n in range(10, 201, 10):
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    y_pred_val = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred_val)
    print(f"n_estimators={n}, RMSE={rmse:.3f}")


n_estimators=10, RMSE=0.460
n_estimators=20, RMSE=0.454
n_estimators=30, RMSE=0.451
n_estimators=40, RMSE=0.448
n_estimators=50, RMSE=0.446
n_estimators=60, RMSE=0.445
n_estimators=70, RMSE=0.445
n_estimators=80, RMSE=0.445
n_estimators=90, RMSE=0.445
n_estimators=100, RMSE=0.444
n_estimators=110, RMSE=0.443
n_estimators=120, RMSE=0.444
n_estimators=130, RMSE=0.443
n_estimators=140, RMSE=0.443
n_estimators=150, RMSE=0.443
n_estimators=160, RMSE=0.443
n_estimators=170, RMSE=0.443
n_estimators=180, RMSE=0.442
n_estimators=190, RMSE=0.443
n_estimators=200, RMSE=0.443


In [ ]:
results = {}

for max_depth in [10, 15, 20, 25]:
    rmse_vals = []
    for n in range(10, 201, 10):
        rf = RandomForestRegressor(max_depth=max_depth, n_estimators=n, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred_val = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred_val)
        rmse_vals.append(rmse)
    mean_rmse = np.mean(rmse_vals)
    results[max_depth] = mean_rmse
    print(f"max_depth={max_depth}, mean RMSE={mean_rmse:.3f}")

best_max_depth = min(results, key=results.get)
print(f"Best max_depth: {best_max_depth}, mean RMSE: {results[best_max_depth]:.3f}")

max_depth=10, mean RMSE=0.442
max_depth=15, mean RMSE=0.445
max_depth=20, mean RMSE=0.446
max_depth=25, mean RMSE=0.446
Best max_depth: 10, mean RMSE: 0.442


In [ ]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
importances = rf.feature_importances_

df_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

df_imp

,feature,importance
13,vehicle_weight,0.959162
6,horsepower,0.016040
0,acceleration,0.011471
3,engine_displacement,0.003269
7,model_year,0.003182
8,num_cylinders,0.002359
9,num_doors,0.001591
12,origin=USA,0.000555
11,origin=Europe,0.000520
10,origin=Asia,0.000476


In [91]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'eval')]

In [92]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1
}

In [93]:
model_03 = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist)

rmse_03 = model_03.eval(dval)
print("RMSE with eta=0.3:", rmse_03)

[0]	train-rmse:1.81393	eval-rmse:1.85444
[5]	train-rmse:0.51381	eval-rmse:0.55664
[10]	train-rmse:0.37115	eval-rmse:0.43896
[15]	train-rmse:0.34666	eval-rmse:0.43362
[20]	train-rmse:0.33553	eval-rmse:0.43376
[25]	train-rmse:0.32268	eval-rmse:0.43683
[30]	train-rmse:0.31475	eval-rmse:0.43752
[35]	train-rmse:0.30960	eval-rmse:0.43784
[40]	train-rmse:0.30202	eval-rmse:0.43968
[45]	train-rmse:0.29126	eval-rmse:0.44024
[50]	train-rmse:0.28456	eval-rmse:0.44140
[55]	train-rmse:0.27618	eval-rmse:0.44225
[60]	train-rmse:0.26768	eval-rmse:0.44290
[65]	train-rmse:0.26174	eval-rmse:0.44352
[70]	train-rmse:0.25489	eval-rmse:0.44531
[75]	train-rmse:0.24792	eval-rmse:0.44628
[80]	train-rmse:0.24254	eval-rmse:0.44689
[85]	train-rmse:0.23644	eval-rmse:0.44749
[90]	train-rmse:0.23193	eval-rmse:0.44839
[95]	train-rmse:0.22475	eval-rmse:0.44904
[99]	train-rmse:0.21950	eval-rmse:0.45018
RMSE with eta=0.3: [0]	eval-rmse:0.45017754981610764


In [94]:
xgb_params['eta'] = 0.1

model_01 = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist)

rmse_01 = model_01.eval(dval)
print("RMSE with eta=0.1:", rmse_01)

[0]	train-rmse:2.28944	eval-rmse:2.34561
[5]	train-rmse:1.41247	eval-rmse:1.44988
[10]	train-rmse:0.91008	eval-rmse:0.94062
[15]	train-rmse:0.63402	eval-rmse:0.66672
[20]	train-rmse:0.48983	eval-rmse:0.53064
[25]	train-rmse:0.41881	eval-rmse:0.46891
[30]	train-rmse:0.38342	eval-rmse:0.44289
[35]	train-rmse:0.36435	eval-rmse:0.43250
[40]	train-rmse:0.35343	eval-rmse:0.42746
[45]	train-rmse:0.34621	eval-rmse:0.42595
[50]	train-rmse:0.33998	eval-rmse:0.42498
[55]	train-rmse:0.33480	eval-rmse:0.42449
[60]	train-rmse:0.33054	eval-rmse:0.42456
[65]	train-rmse:0.32602	eval-rmse:0.42493
[70]	train-rmse:0.32202	eval-rmse:0.42503
[75]	train-rmse:0.31895	eval-rmse:0.42526
[80]	train-rmse:0.31667	eval-rmse:0.42563
[85]	train-rmse:0.31440	eval-rmse:0.42574
[90]	train-rmse:0.31059	eval-rmse:0.42586
[95]	train-rmse:0.30625	eval-rmse:0.42611
[99]	train-rmse:0.30419	eval-rmse:0.42623
RMSE with eta=0.1: [0]	eval-rmse:0.42622800170587150
